# Daily CMIP6 simulations, 1980–2014

Find `rlut`, precipitation (`pr`), and the available pressure-level profiles
of `ua`, `va`, `ta`, and specific humidity (`hus`). Require daily sampling,
full calendar years 1980–2014, and a common simulation and horizontal grid.
Then identify models with qualifying **AMIP and historical** runs.

The complete catalog is loaded once. Searches use that in-memory object.
The pressure check reuses saved header evidence; this notebook does not
scan the archive, create caches, or submit jobs.


In [1]:
import pandas as pd
from IPython.display import display
from cmip_catalog import Catalog, Field, MatchResult, VerticalRequirement, build_filelists, load_catalog
from src.paths import CMIP6_ARCHIVE_DIR, CMIP6_CATALOG_FILE, CMIP6_HEADERS_FILE


## Load the complete catalog once

`cmip6_catalog.parquet` is the existing complete local copy, renamed for
clarity. Run this cell once per kernel; use `.search()` for subsequent queries.
Refreshing the copy is an explicit operation described in the
[workflow guide](../docs/catalog_workflow.md).


In [2]:
catalog = load_catalog("CMIP6", source=CMIP6_CATALOG_FILE)
print(f"{len(catalog.df):,} files in the complete CMIP6 catalog")


7,109,975 files in the complete CMIP6 catalog


## Search in memory

This lists candidate rows, not yet a complete pressure-qualified cohort.
Frequency comes from metadata, independently of table ID. All members are
eligible. The two experiments are matched separately before intersecting models.


In [3]:
# Search catalog for simulations with:
# - rlut and precip
# - ua, va, ta, and hus on available pressure levels
# - daily time resolution
# - AMIP and historical runs from 1980 to 2014
VARIABLES = ["rlut", "pr", "ua", "va", "ta", "hus"]
PROFILE_VARIABLES = ["ua", "va", "ta", "hus"]
EXPERIMENTS = ["amip", "historical"]
TIME_WINDOW = ("1980", "2014")
PREFER_TABLES = ("Eday", "day", "CFday", "EdayZ")

candidates = catalog.search(
    variable_id=VARIABLES, experiment_id=EXPERIMENTS, frequency="daily",
)
display(candidates.summary(by=("experiment_id", "variable_id", "table_id")))


,experiment_id,variable_id,table_id,files
0,amip,hus,CFday,376
1,amip,hus,Eday,947
2,amip,hus,EdayZ,178
3,amip,hus,day,1089
4,amip,pr,day,924
5,amip,rlut,day,358
6,amip,ta,CFday,251
7,amip,ta,Eday,835
8,amip,ta,EdayZ,200
9,amip,ta,day,1127


## Reuse the pressure and horizontal-coordinate evidence

`cmip6_headers.parquet` consolidates the previous header audit for these
fields, experiments and years. It records actual coordinate kinds/values,
calendars, horizontal fingerprints, sample paths, and whether evidence is
direct or propagated from a representative file. It is a **dated study
snapshot**, not an automatically refreshed cache.

The inner join keeps rows covered by that saved evidence. It is not a claim
that all other files are unavailable. A different question may need additional
metadata. The archive symlink routes paths through this project's `data/`
directory, as required by its working conventions; no archive files are copied.


In [4]:
candidates = candidates.remap_paths({
    str(CMIP6_ARCHIVE_DIR.resolve()) + "/": str(CMIP6_ARCHIVE_DIR) + "/",
})
headers = pd.read_parquet(CMIP6_HEADERS_FILE)
frame = candidates.df.merge(headers, on="uri", how="inner", validate="one_to_one")
frame["calendar"] = frame.file_calendar.fillna(frame.calendar)
checked = Catalog(frame)
checked = checked.search(horizontal_grid=checked.unique("horizontal_grid"))
print(f"{len(frame):,} catalog rows have saved header evidence")


17,325 catalog rows have saved header evidence


## Require complete simulations

Require actual multi-level **pressure** coordinates for the four profiles,
equal inspected horizontal fingerprints, and full calendar-aware coverage.
Table preference chooses among feasible alternatives; it does not infer
frequency or vertical coordinates. No members, grids or versions are stitched
together to fill gaps. No interpolation is performed.


In [5]:
fields = [
    Field(v, vertical=VerticalRequirement("pressure", min_levels=2))
    if v in PROFILE_VARIABLES else Field(v)
    for v in VARIABLES
]
complete = checked.match(
    fields, frequency="daily", time_window=TIME_WINDOW, coverage="full",
    prefer_tables=PREFER_TABLES, grid_policy="same",
)
if not complete.matches:
    display(complete.diagnostics)
    raise ValueError("No complete simulations satisfy these requirements")
complete_summary = complete.summary()
display(complete_summary.groupby("experiment_id").agg(
    simulations=("member_id", "size"), models=("source_id", "nunique"),
))
display(complete.diagnostics.reason.value_counts())


,simulations,models
experiment_id,,
amip,55,19
historical,174,25


reason
grid_mismatch        25
incomplete_window    14
Name: count, dtype: int64

## Models available in both experiments

Count complete simulations by model and experiment, then require at least one
in each. Member labels can differ between experiments. Equal labels do not
establish paired physical initial conditions.


In [6]:
MODEL_KEYS = ["project", "institution_id", "source_id"]
availability = (
    complete_summary.groupby(MODEL_KEYS + ["experiment_id"], dropna=False)
    .size().unstack("experiment_id", fill_value=0)
    .reindex(columns=EXPERIMENTS, fill_value=0)
)
models_in_both = availability.loc[availability.gt(0).all(axis=1)]
display(models_in_both)
print(f"{len(models_in_both)} models have qualifying runs in both experiments")

model_keys = {tuple(None if pd.isna(v) else v for v in key) for key in models_in_both.index}
cohort = MatchResult(
    tuple(m for m in complete.matches if tuple(m.simulation[k] for k in MODEL_KEYS) in model_keys),
    complete.diagnostics,
)


18 models have qualifying runs in both experiments


experiment_id                              amip  historical
project institution_id    source_id                        
CMIP6   CCCma             CanESM5             1           1
        CNRM-CERFACS      CNRM-CM6-1          1          18
                          CNRM-CM6-1-HR       1           1
                          CNRM-ESM2-1         1          10
        HAMMOZ-Consortium MPI-ESM-1-2-HAM     3           3
        INM               INM-CM4-8           1           1
                          INM-CM5-0           1          10
        IPSL              IPSL-CM6A-LR       11          31
        MIROC             MIROC6             10           5
        MPI-M             MPI-ESM1-2-HR       3          10
                          MPI-ESM1-2-LR       3          51
        MRI               MRI-ESM2-0          3           5
        NCAR              CESM2               4          10
                          CESM2-FV2           2           3
                          CESM2-WACCM         3           3
                          CESM2-WACCM-FV2     2           3
        NOAA-GFDL         GFDL-CM4            1           1
        SNU               SAM0-UNICON         1           1

## Report the available pressure levels

Retain each variable's available profile. Different arrays remain different,
even if they contain the same number of levels. This table reports hPa and
evidence provenance; it does not place fields on a common vertical grid.


In [7]:
records = []
for match in complete.matches:
    for variable in PROFILE_VARIABLES:
        selection = match.fields[variable]
        metadata = selection.rows.iloc[0]
        records.append({
            **match.simulation, "variable_id": variable, "table_id": selection.table_id,
            "levels_hpa": tuple(p / 100 for p in metadata.pressure_levels_pa),
            "metadata_uri": metadata.metadata_uri, "evidence": selection.vertical_evidence,
        })
profile_levels = pd.DataFrame(records)
display(profile_levels.groupby(["variable_id", "table_id", "levels_hpa"]).size().rename("simulations"))
pressure_grids = profile_levels.groupby(MODEL_KEYS + ["experiment_id", "member_id"], dropna=False).levels_hpa.nunique()
print(f"{pressure_grids.gt(1).sum()} of {len(complete)} simulations use different pressure arrays between variables")


27 of 229 simulations use different pressure arrays between variables


variable_id  table_id  levels_hpa                                                                                                                   
hus          Eday      (1000.0, 925.0, 850.0, 700.0, 600.0, 500.0, 400.0, 300.0, 250.0, 200.0, 150.0, 100.0, 70.0, 50.0, 30.0, 20.0, 10.0, 5.0, 1.0)    136
             day       (1000.0, 850.0, 700.0, 500.0, 250.0, 100.0, 50.0, 10.0)                                                                           93
ta           Eday      (1000.0, 925.0, 850.0, 700.0, 600.0, 500.0, 400.0, 300.0, 250.0, 200.0, 150.0, 100.0, 70.0, 50.0, 30.0, 20.0, 10.0, 5.0, 1.0)    116
             day       (1000.0, 850.0, 700.0, 500.0, 250.0, 100.0, 50.0, 10.0)                                                                          113
ua           Eday      (1000.0, 925.0, 850.0, 700.0, 600.0, 500.0, 400.0, 300.0, 250.0, 200.0, 150.0, 100.0, 70.0, 50.0, 30.0, 20.0, 10.0, 5.0, 1.0)    115
             day       (1000.0, 850.0, 700.0, 500.0, 250.0, 100.0, 50.0

## Build file lists in memory

Use the models represented in both experiments. Nothing is written here.
Each list contains whole files; apply its metadata's common periods when
opening data. `write_filelists(filelists, output_dir)` is an explicit later step.


In [8]:
filelists = build_filelists(cohort)
print(f"{len(cohort)} qualifying simulations; {len(filelists)} variable file lists in memory")


219 qualifying simulations; 1314 variable file lists in memory


## Limits and reference notes

Full coverage means coverage of **catalog file ranges**. Saved representative
headers do not prove every file's existence, unchanged coordinates, internal
timesteps, missing values, cell bounds or vector staggering. Header metadata
is not refreshed automatically. Coordinate fingerprint mismatch may reflect
rounding as well as different grids; rejected cases have not been diagnosed.

- [Simple workflow and file layout](../docs/catalog_workflow.md)
- [Recorded header findings and original audit](../docs/cmip6_header_check.md)
